# Chapter 17 — Versioning the Space

**Book alignment:** Embeddings From First Principles, Chapter 17

**Question this notebook isolates:** A model upgrade is a model swap you performed on
yourself, over a corpus you already stored. What identifies a space (the `space_hash`), what
does a *matching* hash actually promise, and what does a naively mixed v1/v2 index cost? On
RELATE (Wave 3), for two close spaces the naive mixed-index penalty is small (~0.008 nDCG@10)
— and two spaces of different width cannot share an index at all.

In [ ]:
from pathlib import Path
import json
import numpy as np

rng = np.random.default_rng(0)


def find_repo_root(start: Path) -> Path:
    for c in (start, *start.parents):
        if (c / "experiments" / "embeddings-from-first-principles" / "wave1").is_dir():
            return c
    raise RuntimeError("run from a checkout containing experiments/embeddings-from-first-principles")


ROOT = find_repo_root(Path.cwd().resolve())
EXP = ROOT / "experiments" / "embeddings-from-first-principles"


def art(wave, name):
    return json.loads((EXP / wave / "artifacts" / f"{name}.json").read_text())

## 1. The space identity — a hash over everything that changes the geometry

In [ ]:
import hashlib

def space_hash(identity: dict) -> str:
    canon = json.dumps(identity, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(canon.encode()).hexdigest()[:16]

v1 = dict(weights_hash="9f2c", tokenizer_hash="a1b2", dim=768, pooling="mean",
          normalization="l2", instruction_prefix_query="", instruction_prefix_document="",
          max_sequence_length=512, truncation_policy="tail", precision="fp32", post_processing="none")
v2 = dict(v1, weights_hash="7e4d")                 # a minor weights bump -> a different universe

h1, h2 = space_hash(v1), space_hash(v2)
print("v1 space_hash:", h1)
print("v2 space_hash:", h2)
assert h1 != h2
# a derived space (PCA / whitening / bridge output / Matryoshka prefix) gets its OWN hash:
v1_pca256 = dict(v1, post_processing="pca(matrix=c55e,k=256)")
assert space_hash(v1_pca256) != h1
print("identity (exact) -> compatibility (measured) -> usability (a scoped policy call): three layers")

## 2. The mixed-index penalty, and the pairs that cannot mix at all (Wave 3)

In [ ]:
mp = art("wave3", "mixed-index-penalty-curve")["pairs"]
mixable = mp["bge-large(v1) + mxbai-large(v2)"]
print(f"naive mixed v1+v2 index  nDCG@10 {mixable['naive_mixed_ndcg10']:.3f}"
      f"   clean all-v2 index {mixable['clean_all_v2_ndcg10']:.3f}")
print(f"naive penalty            {mixable['naive_penalty']:.4f}")
print(f"residual after per-space calibrated merge  {mixable['residual_penalty_after_merge']:.4f}")
assert mixable["naive_penalty"] > 0                       # even two very close spaces pay something

skipped = [k for k, v in mp.items() if isinstance(v, dict) and str(v.get("status", "")).startswith("SKIPPED")]
print("\npairs that cannot share an index at all (dims differ):")
for k in skipped:
    print("  -", k)
assert len(skipped) == 2
# v1->v2 of the SAME model is usually a width change too -> you cannot even put the vectors
# in one index without a bridge (Chapter 20). the book's hypothesis: the penalty grows as
# v1 and v2 diverge; RELATE v0.1 ships only near pairs, so that stays a hypothesis here.

## 3. Thresholds, calibrations, and evals are all bound to `space_hash`

In [ ]:
registry = {h1: dict(calibration_threshold=0.843, eval_ndcg10=0.952),
            h2: dict(calibration_threshold=None,  eval_ndcg10=None)}   # v2: everything must be re-derived
assert registry[h2]["calibration_threshold"] is None
print("a v1 threshold applied to v2 vectors is half a comparison across universes - re-derive on every bump")

## What we earned

A space is identified by everything in the pipeline that can move the geometry, hashed to a
`space_hash`. A matching hash promises "same declared pipeline" — not "same geometry" and
never "safe to mix". Compatibility is measured (Chapters 16, 21); usability is a scoped
policy call (Chapter 20). Every threshold, calibration, and eval is bound to the hash. A
naively mixed-space index carries a recall penalty even when the two spaces are close, and
different-width spaces cannot share an index without a bridge.

**Notebook 18 / Chapter 18** asks the constructive question: can one space be *translated*
into another, and by what standard does "≈" succeed?